# V2: Tuned XGBoost + 5-Seed Ensemble
## 162-combo grid search + 5-seed XGBoost ensemble

MAE (daytime): 83.01 W/m² | R²: 0.8071


In [ ]:
# Install required packages (run once)
# !pip install pandas numpy xgboost pvlib scikit-learn scikit-learn-extra scipy openpyxl pyarrow tqdm

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
from sklearn.linear_model import QuantileRegressor
import pvlib
from pvlib import location, solarposition, clearsky, temperature, pvsystem
from scipy import stats
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
import json
import hashlib
import warnings
import os
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

# k-medoids: try sklearn_extra first, fall back to built-in implementation
try:
    from sklearn_extra.cluster import KMedoids
    print('Using sklearn_extra KMedoids')
except Exception:
    class KMedoids:
        """Minimal k-medoids (PAM-style alternate) fallback."""
        def __init__(self, n_clusters=5, random_state=None, metric='euclidean', method='alternate'):
            self.n_clusters = n_clusters
            self.random_state = random_state
            self.metric = metric
            self.medoid_indices_ = None
        def fit_predict(self, X):
            rng = np.random.RandomState(self.random_state)
            n = len(X)
            D = pairwise_distances(X, metric=self.metric)
            medoids = rng.choice(n, self.n_clusters, replace=False)
            for _ in range(100):
                labels = np.argmin(D[:, medoids], axis=1)
                new_medoids = np.empty(self.n_clusters, dtype=int)
                for k in range(self.n_clusters):
                    members = np.where(labels == k)[0]
                    if len(members) == 0:
                        new_medoids[k] = medoids[k]
                        continue
                    costs = D[np.ix_(members, members)].sum(axis=1)
                    new_medoids[k] = members[np.argmin(costs)]
                if np.array_equal(new_medoids, medoids):
                    break
                medoids = new_medoids
            self.medoid_indices_ = medoids
            return np.argmin(D[:, medoids], axis=1)
    print('Using built-in KMedoids fallback (sklearn_extra unavailable)')

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

print('All imports OK')

## Configuration

In [ ]:
# ============================================================
# CONFIGURATION — all key parameters centralised here
# ============================================================
CFG = dict(
    # --- Paths ---
    cwa_dir   = 'Project_Archive_Prediction_Final/data/raw/UTC+8 CWA_hourly',
    nwp_dir   = 'Project_Archive_Prediction_Final/data/raw/UTC+0_NWP_grib_extracted',
    load_path = 'Project_Archive_Prediction_Final/data/raw/NTUST_Load_merged_fixed_v2.xlsx',
    output_dir = 'pipeline_outputs',

    # --- Site metadata (NTUST, Taipei) ---
    site_lat  = 25.0377,
    site_lon  = 121.5149,
    site_elev = 10.0,
    tz        = 'Asia/Taipei',

    # --- Time rules ---
    gate_hour_utc = 12,

    # --- Quantiles ---
    quantiles = [round(0.05 + 0.05 * i, 2) for i in range(19)],

    # --- Training ---
    day_training_mode = 'day_only',  # train only on solar_elevation > 0

    # --- XGBoost defaults (overridden by tuning) ---
    xgb_params = dict(
        learning_rate=0.05, max_depth=8, subsample=0.75,
        colsample_bytree=0.8, min_child_weight=5,
    ),
    xgb_rounds     = 3000,
    xgb_early_stop = 50,

    # --- Hyperparameter tuning grid ---
    tune_enabled = True,
    tune_grid = dict(
        learning_rate    = [0.02, 0.035, 0.05],
        max_depth        = [6, 8, 10],
        subsample        = [0.65, 0.75, 0.85],
        colsample_bytree = [0.7, 0.8],
        min_child_weight = [3, 5, 10],
    ),

    # --- PV conversion (PVWatts-style) ---
    pv_dc_rated_kw = 50.0,
    gamma_pdc      = -0.0047,
    inv_eff        = 0.96,
    dc_ac_ratio    = 1.2,
    inoct          = 49.0,

    # --- Scenario generation ---
    N_scenarios = 500,
    K_reduced   = 5,
    random_seed = 42,
    ensemble_seeds = [42, 123, 456, 789, 1024],  # 5-seed ensemble
)

# Derived
CFG['pv_ac_rated_kw'] = CFG['pv_dc_rated_kw'] / CFG['dc_ac_ratio']
CFG['quantile_pairs'] = [(CFG['quantiles'][i], CFG['quantiles'][18 - i]) for i in range(9)]

Path(CFG['output_dir']).mkdir(parents=True, exist_ok=True)
np.random.seed(CFG['random_seed'])

print(f"Training mode: {CFG['day_training_mode']}")
print(f"Tuning enabled: {CFG['tune_enabled']}")
print(f"Quantiles: {CFG['quantiles']}")

---
## S0: Raw Data Ingestion & Cleaning

### S0a: CWA Hourly Observations

In [ ]:
def load_cwa(cwa_dir: str) -> pd.DataFrame:
    """Load & clean all monthly CWA CSVs → hourly DataFrame indexed by ts_local."""
    path = Path(cwa_dir)
    files = sorted(path.glob('*.csv'))
    print(f'Found {len(files)} CWA files')

    frames = []
    for f in tqdm(files, desc='CWA'):
        df = pd.read_csv(f)
        df.columns = [c.strip() for c in df.columns]

        # Parse YYYYMM / DD / HH → ts_local
        year  = df['YYYYMM'].astype(str).str[:4]
        month = df['YYYYMM'].astype(str).str[4:6]
        day   = df['DD'].astype(str).str.zfill(2)
        # HH may contain '00:00' or just '0'
        hour  = df['HH'].astype(str).str.split(':').str[0].str.zfill(2)
        df['ts_local'] = pd.to_datetime(year + '-' + month + '-' + day + ' ' + hour + ':00',
                                        errors='coerce')
        frames.append(df)

    cwa = pd.concat(frames, ignore_index=True)
    cwa = cwa.dropna(subset=['ts_local']).sort_values('ts_local')
    cwa = cwa.drop_duplicates(subset=['ts_local'], keep='last')

    # Clean string anomalies → NaN
    for col in cwa.columns:
        if cwa[col].dtype == object:
            cwa[col] = cwa[col].replace(['None', 'none', '', ' '], np.nan)
            cwa[col] = cwa[col].str.strip() if hasattr(cwa[col], 'str') else cwa[col]

    # Unit conversion: GHI MJ/m² → W/m²  (spec §4.1 step 3)
    cwa['Global_Solar_Radiation_MJm2'] = pd.to_numeric(
        cwa['Global_Solar_Radiation_MJm2'], errors='coerce')
    cwa['ghi_obs_wm2'] = (cwa['Global_Solar_Radiation_MJm2'] * 1_000_000 / 3600).clip(lower=0)

    # Numeric coercion for key fields
    for col in ['Air_Temperature_C', 'Relative_Humidity_percent', 'Wind_Speed_ms',
                'Precipitation_mm', 'Sunshine_Duration_hour', 'Total_Cloud_Amount_tenths']:
        if col in cwa.columns:
            cwa[col] = pd.to_numeric(cwa[col], errors='coerce')

    cwa = cwa.set_index('ts_local')
    # Localise as Asia/Taipei
    cwa.index = cwa.index.tz_localize(CFG['tz'], ambiguous='NaT', nonexistent='NaT')
    cwa = cwa[cwa.index.notna()]

    print(f'CWA shape: {cwa.shape}, range: {cwa.index.min()} → {cwa.index.max()}')
    return cwa

df_cwa = load_cwa(CFG['cwa_dir'])
df_cwa[['ghi_obs_wm2']].describe()

### S0b: NWP Raw Database

In [ ]:
def load_nwp(nwp_dir: str) -> pd.DataFrame:
    """Load all gfs_*_raw_database.csv, merge on (initial_time, valid_time, forecast_hour)."""
    path = Path(nwp_dir)
    files = sorted(path.glob('gfs_*_raw_database.csv'))
    print(f'Found {len(files)} NWP variable files')

    merged = None
    for f in tqdm(files, desc='NWP'):
        var_name = f.stem.replace('gfs_', '').replace('_raw_database', '')
        df = pd.read_csv(f)
        df['initial_time'] = pd.to_datetime(df['initial_time'])
        df['valid_time']   = pd.to_datetime(df['valid_time'])
        df = df.rename(columns={var_name: var_name})  # already named
        # Keep only needed cols
        cols = ['initial_time', 'valid_time', 'forecast_hour', var_name]
        cols = [c for c in cols if c in df.columns]
        df = df[cols]

        if merged is None:
            merged = df
        else:
            merged = merged.merge(df, on=['initial_time', 'valid_time', 'forecast_hour'], how='outer')

    # Deduplicate
    n_before = len(merged)
    merged = merged.drop_duplicates(subset=['initial_time', 'valid_time'], keep='last')
    n_dup = n_before - len(merged)
    print(f'NWP rows: {len(merged)}, duplicates removed: {n_dup}')
    print(f'NWP range: {merged["valid_time"].min()} → {merged["valid_time"].max()}')
    return merged

df_nwp_raw = load_nwp(CFG['nwp_dir'])
df_nwp_raw.head()

### S0c: Load Data

In [ ]:
def load_load_data(path: str) -> pd.DataFrame:
    """Load NTUST hourly load data."""
    df = pd.read_excel(path)
    df['ts_local'] = pd.to_datetime(df['DateTime'])
    df = df[['ts_local', 'Load_kWh']].rename(columns={'Load_kWh': 'load_kw'})
    df = df.set_index('ts_local').sort_index()
    df.index = df.index.tz_localize(CFG['tz'], ambiguous='NaT', nonexistent='NaT')
    df = df[df.index.notna()]
    print(f'Load shape: {df.shape}, range: {df.index.min()} → {df.index.max()}')
    return df

df_load = load_load_data(CFG['load_path'])
df_load.describe()

---
## S1: Gate Compliance

For target day D, deadline = D−1 12:00 UTC (= D−1 20:00 local).
Select `chosen_init_utc = max(initial_time)` where `initial_time <= deadline_utc`.
Extract the 24h window: local D 00:00–23:00 = UTC D−1 16:00 → D 15:00.

In [ ]:
def apply_gate_compliance(df_nwp: pd.DataFrame) -> pd.DataFrame:
    """Gate-filter NWP: for each target day, pick the latest run before the deadline."""
    df = df_nwp.copy()

    # Convert valid_time to local target day
    # local = UTC + 8h; local day boundary 00:00 local = 16:00 UTC previous day
    df['ts_local'] = df['valid_time'] + pd.Timedelta(hours=8)
    df['target_day_local'] = df['ts_local'].dt.normalize()

    # Deadline: D-1 12:00 UTC for target day D
    df['deadline_utc'] = df['target_day_local'] - pd.Timedelta(hours=8) - pd.Timedelta(hours=12)
    # That simplifies to: target_day_local - 20 hours (in UTC)
    # Or equivalently: (D in local) → D-1 20:00 local → D-1 12:00 UTC
    df['deadline_utc'] = df['target_day_local'] - pd.Timedelta(days=1) + pd.Timedelta(hours=12) - pd.Timedelta(hours=8)
    # target_day_local is already naive (represents local midnight), deadline = target_day_local - 1 day + 12h - 8h = target_day_local - 20h
    # Actually: deadline_utc = (D_local - 1day) at 12:00 UTC
    # D_local midnight = D-1 16:00 UTC; D-1 12:00 UTC is 4h before that
    df['deadline_utc'] = df['target_day_local'] - pd.Timedelta(hours=20)  # simple: D_local_midnight - 20h = D-1 04:00 local = D-2 20:00 UTC... 
    # Let me be precise:
    # target_day_local is a naive datetime at midnight local.
    # deadline in local = D-1 20:00 local.
    # deadline in UTC   = D-1 12:00 UTC.
    # Since initial_time is in UTC: deadline_utc = target_day_local - Timedelta(days=1, hours=-12)
    # target_day_local is naive midnight → treat as if UTC for arithmetic, then the
    # actual UTC equivalent of local midnight D is D-1 16:00 UTC.
    # So deadline_utc = D-1 12:00 UTC. 
    # In naive terms: target_day_local represents local D 00:00.
    # D-1 20:00 local = target_day_local - 4 hours (naive local).
    # D-1 12:00 UTC = (D-1 20:00 local) - 8 hours.
    # initial_time is in UTC. So:
    # deadline_utc (as a real UTC timestamp) = target_day_local(naive) - 4h - 8h = target_day_local - 12h
    # But target_day_local(naive) midnight local = midnight of the local day.
    # Actually initial_time is stored as naive UTC datetimes. So to compare correctly:
    # local D midnight = UTC (D-1 16:00).
    # deadline = UTC D-1 12:00.
    # So: deadline_utc = (D_local_midnight_as_naive - 8h) - 4h  ... no.
    # Let's just compute it directly: for target_day_local D,
    # deadline_utc = pd.Timestamp(D) - pd.Timedelta(days=1) + pd.Timedelta(hours=12)
    # But we must convert D from local to UTC first for the comparison.
    # Since initial_time is naive UTC, let's compute deadline as naive UTC:
    # D-1 12:00 UTC. D (local midnight) in UTC = D - 8h. So D-1 12:00 UTC = D - 8h - 12h = D - 20h.
    # Wait: D is a local date. D-1 12:00 UTC is just the date D-1 at 12:00, naive.
    # target_day_local is a pandas Timestamp at midnight (naive). To get D-1 12:00 as naive UTC:
    # deadline_utc_naive = target_day_local - pd.Timedelta(days=1) + pd.Timedelta(hours=12)
    # BUT target_day_local is a LOCAL date. D local midnight in UTC = D - 8h.
    # D-1 12:00 UTC as naive = we need to subtract 1 day from the LOCAL date then add 12h.
    # Since target_day_local is stored as naive datetime at midnight:
    # D_local midnight as naive = e.g. 2024-07-15 00:00:00
    # D-1 = 2024-07-14
    # D-1 12:00 UTC = 2024-07-14 12:00:00 (this is what we want to compare to initial_time which is UTC)
    # So: deadline = target_day_local - Timedelta(days=1) + Timedelta(hours=12)
    # This is correct because the dates are the same concept — just the label of the day.

    df['deadline_utc'] = df['target_day_local'] - pd.Timedelta(days=1) + pd.Timedelta(hours=12)

    # Filter: only runs where initial_time <= deadline
    df_valid = df[df['initial_time'] <= df['deadline_utc']].copy()

    # For each (target_day_local, valid_time), pick the latest initial_time
    df_valid = df_valid.sort_values(['target_day_local', 'valid_time', 'initial_time'],
                                    ascending=[True, True, False])
    gated = df_valid.drop_duplicates(subset=['target_day_local', 'valid_time'], keep='first')

    # Record chosen_init_utc per target day
    chosen = gated.groupby('target_day_local')['initial_time'].max().rename('chosen_init_utc')
    gated = gated.merge(chosen, on='target_day_local', how='left')

    # Compute lead_hour
    gated['lead_hour'] = (gated['valid_time'] - gated['chosen_init_utc']).dt.total_seconds() / 3600

    # Keep only the 24h window for each target day (local 00:00–23:00)
    gated['target_hour_local'] = gated['ts_local'].dt.hour
    gated['target_date_check'] = gated['ts_local'].dt.normalize()
    gated = gated[gated['target_date_check'] == gated['target_day_local']]

    # Retain traceability columns
    gated['ts_utc'] = gated['valid_time']
    gated['valid_time_utc_raw'] = gated['valid_time'].astype(str)

    print(f'Gate-compliant NWP: {len(gated)} rows, '
          f'{gated["target_day_local"].nunique()} target days')

    # Build gate manifest
    manifest = gated.groupby('target_day_local').agg(
        chosen_init_utc=('chosen_init_utc', 'first'),
        deadline_utc=('deadline_utc', 'first'),
        n_hours=('valid_time', 'count'),
    ).reset_index()
    manifest['fallback_count'] = 0
    manifest.to_parquet(f"{CFG['output_dir']}/nwp_gate_manifest.parquet", index=False)

    return gated

df_nwp_gated = apply_gate_compliance(df_nwp_raw)
df_nwp_gated[['target_day_local', 'ts_local', 'ts_utc', 'chosen_init_utc',
              'deadline_utc', 'lead_hour', 'dswrf']].head(10)

---
## S2: Hourly Features

- NWP 3h→1h interpolation (linear for most, TP-A for precipitation)
- Merge CWA + NWP on ts_local
- Add solar position, clear-sky, lag features

In [ ]:
def nwp_3h_to_1h(gated: pd.DataFrame) -> pd.DataFrame:
    """
    Upsample NWP from 3h to 1h per target day.
    - General variables: linear interpolation
    - tp (precipitation): TP-A — evenly distribute 3h totals into 1h bins
    """
    linear_vars = ['dswrf', 'tcc', 'lcc', 'mcc', 'hcc', 't2m', 'u', 'v', 'r2']
    linear_vars = [v for v in linear_vars if v in gated.columns]
    has_tp = 'tp' in gated.columns

    results = []
    for day, grp in tqdm(gated.groupby('target_day_local'), desc='3h→1h'):
        grp = grp.sort_values('ts_local').set_index('ts_local')

        # Build full hourly index for this day
        day_ts = pd.Timestamp(day)
        full_idx = pd.date_range(
            start=day_ts.tz_localize(CFG['tz']),
            periods=24, freq='h', tz=CFG['tz'])

        # Localise index for merge if needed
        if grp.index.tz is None:
            grp.index = grp.index.tz_localize(CFG['tz'])

        # Reindex to hourly
        hourly = grp[linear_vars].reindex(full_idx)
        hourly = hourly.interpolate(method='linear', limit_direction='both')

        # TP-A for precipitation: divide 3h total evenly into 3 × 1h
        if has_tp:
            tp_3h = grp[['tp']].reindex(full_idx)
            # Forward-fill the 3h value, then divide by 3
            tp_3h['tp'] = tp_3h['tp'].ffill().bfill() / 3.0
            hourly['tp'] = tp_3h['tp']

        # Carry forward metadata
        meta_cols = ['target_day_local', 'chosen_init_utc', 'deadline_utc']
        for col in meta_cols:
            if col in grp.columns:
                hourly[col] = grp[col].iloc[0]

        hourly['ts_local'] = hourly.index
        # Compute ts_utc from ts_local
        hourly['ts_utc'] = hourly.index.tz_convert('UTC')
        results.append(hourly)

    nwp_hourly = pd.concat(results)

    # Unit conversions per spec §3.2
    if 'dswrf' in nwp_hourly.columns:
        nwp_hourly['dswrf'] = nwp_hourly['dswrf'].clip(lower=0)
    if 't2m' in nwp_hourly.columns:
        nwp_hourly['t2m_c'] = nwp_hourly['t2m'] - 273.15  # K → °C
    if 'u' in nwp_hourly.columns and 'v' in nwp_hourly.columns:
        nwp_hourly['wind_speed'] = np.sqrt(nwp_hourly['u']**2 + nwp_hourly['v']**2)
    for cc in ['tcc', 'lcc', 'mcc', 'hcc']:
        if cc in nwp_hourly.columns:
            nwp_hourly[cc] = nwp_hourly[cc].clip(0, 100)

    print(f'NWP hourly shape: {nwp_hourly.shape}')
    return nwp_hourly

df_nwp_hourly = nwp_3h_to_1h(df_nwp_gated)
df_nwp_hourly.head()

In [ ]:
def build_features_hourly(cwa: pd.DataFrame, nwp_hourly: pd.DataFrame) -> pd.DataFrame:
    """Merge CWA + NWP, add solar geometry, clear-sky, V6 lag features."""
    nwp = nwp_hourly.set_index('ts_local') if 'ts_local' in nwp_hourly.columns else nwp_hourly
    merged = cwa.join(nwp, how='inner', rsuffix='_nwp')
    print(f'Merged CWA+NWP shape: {merged.shape}')

    # ── Solar geometry (pvlib) ──
    site = location.Location(CFG['site_lat'], CFG['site_lon'],
                             tz=CFG['tz'], altitude=CFG['site_elev'])
    solpos = site.get_solarposition(merged.index)
    merged['solar_elevation'] = solpos['elevation'].values
    merged['solar_zenith']    = solpos['zenith'].values
    merged['solar_azimuth']   = solpos['azimuth'].values

    # ── Clear-sky baseline (Ineichen) — physics only, no obs ──
    cs = site.get_clearsky(merged.index, model='ineichen')
    merged['ghi_clear_wm2'] = cs['ghi'].values
    merged['daylight_mask'] = (merged['solar_elevation'] > 0).astype(int)

    # ── NWP-based clear-sky ratio (gate-safe: uses NWP, not obs) ──
    if 'dswrf' in merged.columns:
        merged['nwp_clear_ratio'] = np.where(
            merged['ghi_clear_wm2'] > 10,
            merged['dswrf'] / merged['ghi_clear_wm2'],
            0.0)

    # ── Gate-safe GHI observation lags ──
    # Gate = D-1 20:00 local. For target hour H on day D:
    #   lag24 = D-1:H. Safe when H <= 20, leaks when H > 20.
    # Fix: for H > 20, use obs at D-1:20 (last pre-gate observation).
    merged['_ghi_lag24_raw'] = merged['ghi_obs_wm2'].shift(24)
    merged['_ghi_lag48_raw'] = merged['ghi_obs_wm2'].shift(48)
    merged['_ghi_lag72_raw'] = merged['ghi_obs_wm2'].shift(72)

    # Build gate obs lookup: obs at hour 20 of previous day
    merged['_hour'] = merged.index.hour
    merged['_date'] = merged.index.normalize()
    _gate_obs = merged[merged['_hour'] == 20][['ghi_obs_wm2']].copy()
    _gate_obs.columns = ['_ghi_at_gate']
    _gate_obs['_date'] = _gate_obs.index.normalize()
    _gate_obs = _gate_obs.set_index('_date')['_ghi_at_gate']
    merged['_prev_date'] = merged['_date'] - pd.Timedelta(days=1)
    merged['_gate_val'] = merged['_prev_date'].map(_gate_obs)

    # Apply gate-safe replacement for hours 21-23
    merged['ghi_obs_lag24'] = np.where(merged['_hour'] <= 20,
                                        merged['_ghi_lag24_raw'],
                                        merged['_gate_val'])
    merged['ghi_obs_lag48'] = merged['_ghi_lag48_raw']  # always safe
    merged['ghi_obs_lag72'] = merged['_ghi_lag72_raw']  # always safe
    merged['ghi_obs_roll3_mean'] = merged[['ghi_obs_lag24', 'ghi_obs_lag48', 'ghi_obs_lag72']].mean(axis=1)
    merged['ghi_obs_roll2_mean'] = merged[['ghi_obs_lag24', 'ghi_obs_lag48']].mean(axis=1)

    # ── CWA observation lag features (gate-safe: all are t-24h+) ──
    OBS_LAG_COLS = ['Sunshine_Duration_hour', 'Total_Cloud_Amount_tenths',
                    'Air_Temperature_C', 'Relative_Humidity_percent',
                    'Wind_Speed_ms', 'Precipitation_mm', 'Visibility_km',
                    'Station_Pressure_hPa']
    for col in OBS_LAG_COLS:
        if col in merged.columns:
            merged[col] = pd.to_numeric(merged[col], errors='coerce').fillna(0)
            raw_lag24 = merged[col].shift(24)
            raw_lag48 = merged[col].shift(48)
            # Gate-safe: replace hours 21-23 with hour-20 value
            _obs_gate = merged[merged['_hour'] == 20][[col]].copy()
            _obs_gate.columns = ['_val']
            _obs_gate['_date'] = _obs_gate.index.normalize()
            _obs_gate = _obs_gate.set_index('_date')['_val']
            _gate_mapped = merged['_prev_date'].map(_obs_gate)
            merged[f'{col}_lag24'] = np.where(merged['_hour'] <= 20,
                                               raw_lag24, _gate_mapped)
            merged[f'{col}_lag48'] = raw_lag48  # always safe
            if col in ['Sunshine_Duration_hour', 'Total_Cloud_Amount_tenths']:
                merged[f'{col}_roll2_mean'] = merged[[f'{col}_lag24', f'{col}_lag48']].mean(axis=1)

    # Clean up temp columns
    merged = merged.drop(columns=[c for c in merged.columns if c.startswith('_')], errors='ignore')

    # ── NWP lag features ──
    if 'dswrf' in merged.columns:
        merged['dswrf_lag24'] = merged['dswrf'].shift(24)

    # ── Calendar / cyclical features ──
    merged['hour'] = merged.index.hour
    merged['month'] = merged.index.month
    merged['doy'] = merged.index.dayofyear
    merged['hour_sin'] = np.sin(2 * np.pi * merged['hour'] / 24)
    merged['hour_cos'] = np.cos(2 * np.pi * merged['hour'] / 24)
    merged['month_sin'] = np.sin(2 * np.pi * merged['month'] / 12)
    merged['month_cos'] = np.cos(2 * np.pi * merged['month'] / 12)

    # Fill NaN from lag shifts
    merged = merged.fillna(0)

    print(f'features_hourly shape: {merged.shape}')
    print(f'Columns: {list(merged.columns)}')
    return merged

features_hourly = build_features_hourly(df_cwa, df_nwp_hourly)
features_hourly.to_parquet(f"{CFG['output_dir']}/features_hourly.parquet")
features_hourly.head()

---
## S3: Issue-Day × Target-Hour Supervised Dataset

One row = one target hour for one issue day.
Time-series split only; random split is forbidden.

In [ ]:
def build_issue_target_dataset(fh: pd.DataFrame) -> pd.DataFrame:
    """Build supervised dataset: each row = (issue_day, target_hour)."""
    df = fh.copy()
    df = df[df.index.notna()]

    # Ensure index is tz-aware; strip tz for clean date arithmetic
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)

    # Derive issue_day_local = target_day - 1 (the day the forecast is made)
    if 'target_day_local' not in df.columns:
        df['target_day_local'] = df.index.normalize()
    df['target_day_local'] = pd.to_datetime(df['target_day_local'])
    # Strip tz from target_day_local if it has one
    if hasattr(df['target_day_local'].dtype, 'tz') and df['target_day_local'].dtype.tz is not None:
        df['target_day_local'] = df['target_day_local'].dt.tz_localize(None)

    df['issue_day_local'] = df['target_day_local'] - pd.Timedelta(days=1)
    df['target_time_local'] = df.index
    df['issue_time_local'] = df['issue_day_local'] + pd.Timedelta(hours=20)  # gate time
    df['horizon_hour'] = (df['target_time_local'] - df['issue_time_local']).dt.total_seconds() / 3600

    # Label
    df['label_ghi_obs_wm2'] = df['ghi_obs_wm2']

    print(f'Dataset shape: {df.shape}')
    print(f'Issue days: {df["issue_day_local"].nunique()}')
    return df

dataset = build_issue_target_dataset(features_hourly)
dataset.head()

In [ ]:
def make_time_splits(dataset: pd.DataFrame) -> dict:
    """
    Chronological split: train / validation / calibration / test.
    Last ~365 days as test. Pre-test: 70/15/15.
    """
    issue_days = sorted(dataset['issue_day_local'].dropna().unique())
    n = len(issue_days)
    print(f'Total issue days: {n}')
    print(f'Range: {issue_days[0]} -> {issue_days[-1]}')

    issue_days_ts = [pd.Timestamp(d).tz_localize(None) if hasattr(pd.Timestamp(d), 'tz') and pd.Timestamp(d).tz is not None else pd.Timestamp(d) for d in issue_days]

    last_day = issue_days_ts[-1]
    test_start = last_day - pd.Timedelta(days=364)

    pre_test = [d for d in issue_days_ts if d < test_start]
    test = [d for d in issue_days_ts if d >= test_start]

    if len(pre_test) < 100:
        n_train = int(n * 0.60)
        n_val   = int(n * 0.15)
        n_cal   = int(n * 0.10)
        splits = {
            'train': issue_days_ts[:n_train],
            'valid': issue_days_ts[n_train:n_train+n_val],
            'calib': issue_days_ts[n_train+n_val:n_train+n_val+n_cal],
            'test':  issue_days_ts[n_train+n_val+n_cal:],
        }
    else:
        n_pre = len(pre_test)
        n_train = int(n_pre * 0.70)
        n_val   = int(n_pre * 0.15)
        splits = {
            'train': pre_test[:n_train],
            'valid': pre_test[n_train:n_train+n_val],
            'calib': pre_test[n_train+n_val:],
            'test':  test,
        }

    split_map = {}
    for name, days in splits.items():
        for d in days:
            split_map[d] = name

    issue_col = dataset['issue_day_local'].copy()
    if hasattr(issue_col.dtype, 'tz') and issue_col.dtype.tz is not None:
        issue_col = issue_col.dt.tz_localize(None)
    dataset['split_name'] = issue_col.map(split_map)

    manifest_rows = []
    for name, days in splits.items():
        manifest_rows.append({
            'split_name': name,
            'start_issue_day_local': min(days),
            'end_issue_day_local':   max(days),
            'n_issue_days': len(days),
            'n_rows': dataset[dataset['split_name'] == name].shape[0],
            'chosen_rule': 'last_365d_test' if len(pre_test) >= 100 else 'proportional_split',
        })
    split_manifest = pd.DataFrame(manifest_rows)
    split_manifest.to_parquet(f"{CFG['output_dir']}/split_manifest.parquet", index=False)

    for name in ['train', 'valid', 'calib', 'test']:
        n_d = len(splits[name])
        n_r = dataset[dataset['split_name'] == name].shape[0]
        print(f'  {name:6s}: {n_d:4d} issue days, {n_r:6d} rows')

    return splits

splits = make_time_splits(dataset)
dataset.to_parquet(f"{CFG['output_dir']}/dataset_issue_target.parquet")


---
## S4: XGBQ 19-Quantile Base Forecast

One model per quantile, pinball loss, validation-based early stopping.
Monotone rearrangement to fix crossing.

In [ ]:
def get_feature_cols(dataset: pd.DataFrame) -> list:
    """Select numeric feature columns — strict day-ahead compliance.
    All contemporaneous CWA observations are excluded since they
    would not be available at forecast time (D-1 20:00 local).
    Only NWP forecasts + lagged observations + physics features allowed.
    """
    # Columns to ALWAYS drop (label, metadata, string/datetime columns)
    drop_always = {
        'label_ghi_obs_wm2', 'ghi_obs_wm2',
        'Global_Solar_Radiation_MJm2',
        'issue_day_local', 'target_day_local', 'target_time_local',
        'issue_time_local', 'ts_utc', 'chosen_init_utc', 'deadline_utc',
        'split_name', 'YYYYMM', 'DD', 'HH', 'Station_No',
    }

    # ALL contemporaneous CWA observations — not available at forecast time
    drop_contemporaneous_obs = {
        'Sunshine_Duration_hour',
        'Total_Cloud_Amount_tenths',
        'Air_Temperature_C',
        'Relative_Humidity_percent',
        'Wind_Speed_ms',
        'Wind_Direction_degree',
        'Gust_Speed_ms',
        'Gust_Direction_degree',
        'Precipitation_mm',
        'Visibility_km',
        'Station_Pressure_hPa',
        'Sea_Level_Pressure_hPa',
    }

    drop_set = drop_always | drop_contemporaneous_obs

    feature_cols = []
    for c in dataset.columns:
        if c in drop_set:
            continue
        if not pd.api.types.is_numeric_dtype(dataset[c]):
            continue
        feature_cols.append(c)

    print(f'Feature columns ({len(feature_cols)}):')
    for c in feature_cols:
        print(f'  {c}')

    # Leakage diagnostic
    label = dataset['label_ghi_obs_wm2']
    print(f'\nLeakage check (|r| > 0.90 with label):')
    suspicious = False
    for c in feature_cols:
        r = dataset[c].corr(label)
        if abs(r) > 0.90:
            print(f'  WARNING: {c} has r = {r:.4f}')
            suspicious = True
    if not suspicious:
        print('  No suspicious correlations found.')

    print(f'\nTotal features: {len(feature_cols)}')
    return feature_cols

feature_cols = get_feature_cols(dataset)


### Hyperparameter Tuning (Grid Search on Q0.50)

Searches over learning_rate, max_depth, subsample, colsample_bytree, min_child_weight
using the validation set and pinball loss at Q0.50. Best params are then used for all 19 quantiles.

In [ ]:
import itertools

def tune_xgb_hyperparams(dataset: pd.DataFrame, feature_cols: list) -> dict:
    """
    Grid search over XGBoost hyperparams using Q0.50 pinball loss on validation set.
    Supports day_only training mode.
    """
    if not CFG.get('tune_enabled', False):
        print('Tuning disabled — using default xgb_params.')
        return CFG['xgb_params']

    train_mask = dataset['split_name'] == 'train'
    valid_mask = dataset['split_name'] == 'valid'

    # Apply day_only filter if enabled
    if CFG.get('day_training_mode') == 'day_only':
        train_mask = train_mask & (dataset['solar_elevation'] > 0)
        valid_mask = valid_mask & (dataset['solar_elevation'] > 0)
        print('Day-only training mode enabled for tuning.')

    X_train = dataset.loc[train_mask, feature_cols].fillna(0)
    y_train = dataset.loc[train_mask, 'label_ghi_obs_wm2'].fillna(0)
    X_valid = dataset.loc[valid_mask, feature_cols].fillna(0)
    y_valid = dataset.loc[valid_mask, 'label_ghi_obs_wm2'].fillna(0)

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)
    print(f'Tuning on {len(X_train)} train / {len(X_valid)} valid samples')

    grid = CFG['tune_grid']
    keys = list(grid.keys())
    combos = list(itertools.product(*[grid[k] for k in keys]))
    print(f'Grid search: {len(combos)} combinations')
    print(f'{"LR":>6s} {"Dep":>4s} {"Sub":>5s} {"Col":>5s} {"MCW":>4s} | {"PB50":>8s} {"MAE":>8s} {"Rnd":>5s}')
    print('-' * 55)

    best_loss = float('inf')
    best_params = {}
    best_rounds = 1000

    for combo in combos:
        trial_params = dict(zip(keys, combo))
        params = {
            'objective': 'reg:quantileerror',
            'quantile_alpha': 0.5,
            'tree_method': 'hist',
            **trial_params,
        }
        model = xgb.train(
            params, dtrain,
            num_boost_round=CFG['xgb_rounds'],
            evals=[(dvalid, 'valid')],
            early_stopping_rounds=CFG['xgb_early_stop'],
            verbose_eval=False,
        )
        pred = model.predict(dvalid)
        err = y_valid.values - pred
        pb = np.mean(np.where(err >= 0, 0.5 * err, -0.5 * err))
        mae = np.mean(np.abs(err))
        n_rnd = model.best_iteration
        lr  = trial_params['learning_rate']
        dep = trial_params['max_depth']
        sub = trial_params['subsample']
        col = trial_params['colsample_bytree']
        mcw = trial_params['min_child_weight']
        print(f'{lr:6.3f} {dep:4d} {sub:5.2f} {col:5.2f} {mcw:4d} | {pb:8.2f} {mae:8.2f} {n_rnd:5d}')
        if pb < best_loss:
            best_loss = pb
            best_params = trial_params.copy()
            best_rounds = n_rnd

    print('-' * 55)
    print(f'Best pinball Q0.50 = {best_loss:.4f}')
    print(f'Best params: {best_params}')
    print(f'Best rounds: {best_rounds}')

    CFG['xgb_params'] = best_params
    CFG['xgb_rounds'] = best_rounds + 300  # headroom for other quantiles
    print(f'\nCFG updated: xgb_params={CFG["xgb_params"]}, xgb_rounds={CFG["xgb_rounds"]}')
    return best_params

tuned_params = tune_xgb_hyperparams(dataset, feature_cols)

In [ ]:
def train_xgbq_19q(dataset: pd.DataFrame, splits: dict, feature_cols: list) -> pd.DataFrame:
    """Train 19 XGBQ models × 5 seeds (multi-seed ensemble)."""
    train_mask = dataset['split_name'] == 'train'
    valid_mask = dataset['split_name'] == 'valid'
    eval_mask  = dataset['split_name'].isin(['calib', 'test'])

    if CFG.get('day_training_mode') == 'day_only':
        train_mask = train_mask & (dataset['solar_elevation'] > 0)
        valid_mask = valid_mask & (dataset['solar_elevation'] > 0)
        print('Day-only training mode: filtering to solar_elevation > 0')

    X_train = dataset.loc[train_mask, feature_cols].fillna(0)
    y_train = dataset.loc[train_mask, 'label_ghi_obs_wm2'].fillna(0)
    X_valid = dataset.loc[valid_mask, feature_cols].fillna(0)
    y_valid = dataset.loc[valid_mask, 'label_ghi_obs_wm2'].fillna(0)
    X_eval  = dataset.loc[eval_mask,  feature_cols].fillna(0)

    print(f'Train: {len(X_train)}, Valid: {len(X_valid)}, Eval: {len(X_eval)}')

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)
    deval  = xgb.DMatrix(X_eval)

    mono_map = {
        'dswrf': 1, 'solar_elevation': 1, 'ghi_clear_wm2': 1,
        'dswrf_lag24': 1, 'nwp_clear_ratio': 1,
        'tcc': -1, 'lcc': -1, 'mcc': -1, 'hcc': -1,
    }
    mono_str = '(' + ','.join(str(mono_map.get(f, 0)) for f in feature_cols) + ')'

    quantiles = CFG['quantiles']
    seeds = CFG.get('ensemble_seeds', [42])
    n_seeds = len(seeds)
    print(f'Multi-seed ensemble: {n_seeds} seeds {seeds}')

    q_cols = [f'q{q:.2f}' for q in quantiles]
    preds_accum = {col: np.zeros(len(X_eval)) for col in q_cols}

    for seed_idx, seed in enumerate(seeds):
        print(f'\n--- Seed {seed} ({seed_idx+1}/{n_seeds}) ---')
        for q in tqdm(quantiles, desc=f'  Seed {seed}'):
            params = {
                'objective': 'reg:quantileerror',
                'quantile_alpha': q,
                'tree_method': 'hist',
                'monotone_constraints': mono_str,
                'seed': seed,
                **CFG['xgb_params'],
            }
            model = xgb.train(
                params, dtrain, num_boost_round=CFG['xgb_rounds'],
                evals=[(dvalid, 'valid')],
                early_stopping_rounds=CFG['xgb_early_stop'],
                verbose_eval=False,
            )
            preds_accum[f'q{q:.2f}'] += model.predict(deval)

    for col in q_cols:
        preds_accum[col] /= n_seeds

    forecast = dataset.loc[eval_mask, [
        'issue_day_local', 'target_day_local', 'target_time_local',
        'horizon_hour', 'label_ghi_obs_wm2', 'split_name',
        'solar_elevation', 'ghi_clear_wm2',
    ]].copy()
    if 'chosen_init_utc' in dataset.columns:
        forecast['chosen_init_utc'] = dataset.loc[eval_mask, 'chosen_init_utc'].values
    if 'deadline_utc' in dataset.columns:
        forecast['deadline_utc'] = dataset.loc[eval_mask, 'deadline_utc'].values

    for col in q_cols:
        forecast[col] = preds_accum[col]

    raw_vals = forecast[q_cols].values
    sorted_vals = np.sort(raw_vals, axis=1)
    crossing_count = int((np.diff(raw_vals, axis=1) < 0).sum())
    forecast[q_cols] = sorted_vals
    print(f'\nCrossing count (before fix): {crossing_count}')

    night_mask = forecast['solar_elevation'] <= 0
    for c in q_cols:
        forecast[c] = forecast[c].clip(lower=0)
        forecast.loc[night_mask, c] = 0.0

    print(f'Ensemble forecast shape: {forecast.shape}')
    return forecast

forecast_raw = train_xgbq_19q(dataset, splits, feature_cols)
forecast_raw.to_parquet(f"{CFG['output_dir']}/forecast_ghi_quantiles_daily_base_raw.parquet")
forecast_raw.head()


---
## S4b: Split-Conformal CQR (Final Calibration Layer)

- 9 symmetric pairs: (0.05,0.95), (0.10,0.90), ..., (0.45,0.55)
- q50 passes through as-is
- Conformity score: s_i(τ) = max(q_low_raw - y, y - q_high_raw)
- Δ_τ = empirical quantile of s at level ceil((n_cal+1)·(1−2τ))/n_cal
- No stacking of QMAP/TOD/regime widening on top

In [ ]:
def apply_cqr(forecast_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Split-conformal CQR on the calibration split.
    Apply offsets to produce calibrated quantiles.
    """
    quantiles = CFG['quantiles']
    q_cols = [f'q{q:.2f}' for q in quantiles]
    pairs = CFG['quantile_pairs']  # [(0.05,0.95), (0.10,0.90), ...]

    cal = forecast_raw[forecast_raw['split_name'] == 'calib'].copy()
    test = forecast_raw[forecast_raw['split_name'] == 'test'].copy()
    y_cal = cal['label_ghi_obs_wm2'].values
    n_cal = len(cal)
    print(f'Calibration samples: {n_cal}')

    deltas = {}
    for (tau_low, tau_high) in pairs:
        col_low  = f'q{tau_low:.2f}'
        col_high = f'q{tau_high:.2f}'

        q_low_raw  = cal[col_low].values
        q_high_raw = cal[col_high].values

        # Conformity score
        scores = np.maximum(q_low_raw - y_cal, y_cal - q_high_raw)

        # Nominal coverage = 1 - 2*tau_low (e.g., for 0.05/0.95 pair, coverage = 0.90)
        alpha = 2 * tau_low  # miscoverage rate
        # Finite-sample correction
        level = np.ceil((n_cal + 1) * (1 - alpha)) / n_cal
        level = min(level, 1.0)
        delta = np.quantile(scores, level)
        deltas[(tau_low, tau_high)] = delta
        print(f'  Pair ({tau_low:.2f}, {tau_high:.2f}): delta = {delta:.2f}')

    # Apply offsets to test set (and calib for diagnostics)
    calibrated = forecast_raw.copy()
    for (tau_low, tau_high), delta in deltas.items():
        col_low  = f'q{tau_low:.2f}'
        col_high = f'q{tau_high:.2f}'
        calibrated[col_low]  = calibrated[col_low]  - delta
        calibrated[col_high] = calibrated[col_high] + delta

    # q50 stays as-is (raw XGBQ median)
    # Monotone rearrangement after CQR
    vals = calibrated[q_cols].values
    calibrated[q_cols] = np.sort(vals, axis=1)

    # Physical clipping
    night_mask = calibrated['solar_elevation'] <= 0
    for c in q_cols:
        calibrated[c] = calibrated[c].clip(lower=0)
        calibrated.loc[night_mask, c] = 0.0
        # Clear-sky upper cap
        if 'ghi_clear_wm2' in calibrated.columns:
            cap = calibrated['ghi_clear_wm2'] * 1.2  # 20% tolerance
            calibrated[c] = calibrated[c].clip(upper=cap)

    # Re-sort after clipping
    calibrated[q_cols] = np.sort(calibrated[q_cols].values, axis=1)

    crossing_count = int((np.diff(calibrated[q_cols].values, axis=1) < 0).sum())
    print(f'Post-CQR crossing count: {crossing_count}')

    calibrated['calibration_method'] = 'split_conformal_cqr'
    calibrated['source'] = 'cqr'
    return calibrated

forecast_cqr = apply_cqr(forecast_raw)
forecast_cqr.to_parquet(f"{CFG['output_dir']}/forecast_ghi_quantiles_daily.parquet")

# Show test-set only
test_fc = forecast_cqr[forecast_cqr['split_name'] == 'test']
print(f'Test forecast: {test_fc.shape}')
test_fc.head()

---
## S5: Load 19Q (Baseline + Residual Quantiles)

Simple approach: `L^(q) = Lbar + eps^(q)` where `eps = Lobs - Lbar`,
and Lbar is the hourly mean load (by hour-of-day, optionally by weekday/season).

In [ ]:
def build_load_quantiles(df_load: pd.DataFrame) -> pd.DataFrame:
    """
    Build load 19Q marginals using deterministic baseline + residual quantiles.
    """
    load = df_load.copy()
    load['hour'] = load.index.hour
    load['dayofweek'] = load.index.dayofweek
    load['is_weekday'] = (load['dayofweek'] < 5).astype(int)

    # Baseline: mean load by (hour, is_weekday)
    baseline = load.groupby(['hour', 'is_weekday'])['load_kw'].mean()
    load['baseline_load_kw'] = load.apply(
        lambda r: baseline.get((r['hour'], r['is_weekday']), r['load_kw']), axis=1)

    # Residual
    load['residual'] = load['load_kw'] - load['baseline_load_kw']

    # Quantiles of residual by (hour, is_weekday)
    quantiles = CFG['quantiles']
    q_cols = [f'load_q{q:.2f}' for q in quantiles]

    residual_quantiles = load.groupby(['hour', 'is_weekday'])['residual'].quantile(quantiles).unstack()
    residual_quantiles.columns = q_cols

    # Build output: for each hour, L^(q) = baseline + eps^(q)
    results = []
    for (hour, is_wd), grp in load.groupby(['hour', 'is_weekday']):
        row = {'hour': hour, 'is_weekday': is_wd,
               'baseline_load_kw': baseline.get((hour, is_wd), 0)}
        rq = residual_quantiles.loc[(hour, is_wd)]
        for i, q in enumerate(quantiles):
            row[f'load_q{q:.2f}'] = row['baseline_load_kw'] + rq.iloc[i]
        results.append(row)

    load_q = pd.DataFrame(results).drop_duplicates(subset=['hour', 'is_weekday'])
    load_q['residual_bucket'] = load_q.apply(
        lambda r: f"h{int(r['hour']):02d}_wd{int(r['is_weekday'])}", axis=1)

    print(f'Load quantiles shape: {load_q.shape}')
    load_q.to_parquet(f"{CFG['output_dir']}/load_quantiles_hourly.parquet", index=False)
    return load_q

load_quantiles = build_load_quantiles(df_load)
load_quantiles.head()

---
## S6: Joint Scenario Generation (Gaussian Copula)

1. For each target hour, build piecewise linear quantile functions from 19Q
2. PIT: u = F(y), clip to [1e-6, 1-1e-6], z = Φ⁻¹(u)
3. Estimate Σ from z vectors; if not PSD → nearest-PSD correction
4. Sample N daily paths from MVN, inverse-map back to physical units
5. Output raw joint (GHI+Load) daily scenarios

In [ ]:
def nearest_psd(A: np.ndarray) -> np.ndarray:
    """Find nearest positive semi-definite matrix."""
    B = (A + A.T) / 2
    eigvals, eigvecs = np.linalg.eigh(B)
    eigvals = np.maximum(eigvals, 1e-8)
    return eigvecs @ np.diag(eigvals) @ eigvecs.T


def quantile_function(quantiles: list, q_values: np.ndarray):
    """Build a piecewise linear quantile function Q(u) from 19Q values."""
    # Extend to 0 and 1 for extrapolation
    taus = np.array([0.0] + list(quantiles) + [1.0])
    vals = np.concatenate([[q_values[0]], q_values, [q_values[-1]]])
    return interp1d(taus, vals, kind='linear', bounds_error=False,
                    fill_value=(vals[0], vals[-1]))


def inverse_quantile(quantiles: list, q_values: np.ndarray):
    """Build inverse quantile function F(y) → u (PIT)."""
    taus = np.array(quantiles)
    vals = np.array(q_values)
    # Handle flat regions
    unique_mask = np.concatenate([[True], np.diff(vals) > 1e-10])
    if unique_mask.sum() < 2:
        return lambda y: np.full_like(y, 0.5, dtype=float)
    return interp1d(vals[unique_mask], taus[unique_mask], kind='linear',
                    bounds_error=False, fill_value=(taus[0], taus[-1]))


print('Copula helper functions defined')

In [ ]:
def generate_joint_scenarios(
    forecast_ghi: pd.DataFrame,
    load_q: pd.DataFrame,
    target_days: list = None,
    N: int = 500,
) -> pd.DataFrame:
    """
    Generate N joint (GHI+Load) daily scenarios via Gaussian Copula.
    """
    quantiles = CFG['quantiles']
    ghi_q_cols  = [f'q{q:.2f}' for q in quantiles]
    load_q_cols = [f'load_q{q:.2f}' for q in quantiles]

    # Use test split only
    fc = forecast_ghi[forecast_ghi['split_name'] == 'test'].copy()
    if target_days is not None:
        fc = fc[fc['target_day_local'].isin(target_days)]

    all_scenarios = []
    psd_fix_count = 0

    for day, grp in tqdm(fc.groupby('target_day_local'), desc='Copula'):
        grp = grp.sort_values('horizon_hour')
        n_hours = len(grp)
        if n_hours < 24:
            continue  # skip incomplete days
        grp = grp.head(24)

        # --- Build marginals ---
        # GHI marginals from CQR forecast
        ghi_qfuncs = []   # Q(u) → GHI
        ghi_inv_funcs = [] # F(y) → u
        for _, row in grp.iterrows():
            q_vals = row[ghi_q_cols].values.astype(float)
            ghi_qfuncs.append(quantile_function(quantiles, q_vals))
            ghi_inv_funcs.append(inverse_quantile(quantiles, q_vals))

        # Load marginals (match by hour and weekday)
        day_ts = pd.Timestamp(day)
        is_wd = int(day_ts.dayofweek < 5)
        load_qfuncs = []
        load_inv_funcs = []
        for h in range(24):
            lq_row = load_q[(load_q['hour'] == h) & (load_q['is_weekday'] == is_wd)]
            if len(lq_row) == 0:
                lq_row = load_q[load_q['hour'] == h]
            if len(lq_row) == 0:
                # Fallback: uniform
                load_qfuncs.append(lambda u: np.full_like(u, 2000.0))
                load_inv_funcs.append(lambda y: np.full_like(y, 0.5))
                continue
            lq_vals = lq_row[load_q_cols].values.flatten().astype(float)
            load_qfuncs.append(quantile_function(quantiles, lq_vals))
            load_inv_funcs.append(inverse_quantile(quantiles, lq_vals))

        # --- PIT on historical data (use the forecast itself for PIT) ---
        # Construct z vectors for correlation estimation
        # For GHI: use the label (observed) mapped through the forecast CDF
        z_ghi = []
        y_obs = grp['label_ghi_obs_wm2'].values
        for h_idx in range(24):
            u = ghi_inv_funcs[h_idx](np.array([y_obs[h_idx]]))
            u = np.clip(u, 1e-6, 1 - 1e-6)
            z_ghi.append(stats.norm.ppf(u)[0])

        # For Load: use baseline as "observation" proxy
        z_load = []
        for h_idx in range(24):
            lq_row = load_q[(load_q['hour'] == h_idx) & (load_q['is_weekday'] == is_wd)]
            if len(lq_row) == 0:
                z_load.append(0.0)
                continue
            baseline = lq_row['baseline_load_kw'].values[0]
            u = load_inv_funcs[h_idx](np.array([baseline]))
            u = np.clip(u, 1e-6, 1 - 1e-6)
            z_load.append(stats.norm.ppf(u)[0])

        z_all = np.array(z_ghi + z_load)  # 48-dim

        # For a single day, use identity correlation (or a simple structure)
        # In practice, Σ should be estimated from many days.
        # Here we use a block-diagonal with mild autocorrelation.
        dim = 48
        Sigma = np.eye(dim)
        # Add temporal autocorrelation within GHI (hours 0-23) and Load (hours 24-47)
        for block_start in [0, 24]:
            for i in range(24):
                for j in range(24):
                    dist = abs(i - j)
                    Sigma[block_start+i, block_start+j] = np.exp(-0.3 * dist)
        # Cross-correlation between GHI and Load
        for i in range(24):
            Sigma[i, 24+i] = 0.3
            Sigma[24+i, i] = 0.3

        # Nearest PSD check
        eigvals = np.linalg.eigvalsh(Sigma)
        if eigvals.min() < 0:
            Sigma = nearest_psd(Sigma)
            psd_fix_count += 1

        # --- Sample N scenarios ---
        rng = np.random.default_rng(CFG['random_seed'] + hash(str(day)) % 10000)
        z_samples = rng.multivariate_normal(np.zeros(dim), Sigma, size=N)

        # Inverse map: z → u → physical
        u_samples = stats.norm.cdf(z_samples)  # (N, 48)

        for s in range(N):
            for h in range(24):
                ghi_val  = float(ghi_qfuncs[h](u_samples[s, h]))
                load_val = float(load_qfuncs[h](u_samples[s, 24 + h]))
                all_scenarios.append({
                    'issue_day_local':  day - pd.Timedelta(days=1),
                    'target_day_local': day,
                    'target_time_local': pd.Timestamp(day) + pd.Timedelta(hours=h),
                    'horizon_hour': h + 4,  # approximate
                    'scenario_id': s,
                    'ghi_wm2': max(ghi_val, 0),
                    'load_kw': max(load_val, 0),
                })

    scenarios = pd.DataFrame(all_scenarios)
    print(f'Raw joint scenarios: {scenarios.shape}, PSD fixes: {psd_fix_count}')
    return scenarios

# For demo, generate for a subset of test days
test_days = sorted(forecast_cqr[forecast_cqr['split_name'] == 'test']['target_day_local'].unique())
demo_days = test_days[:7]  # first week of test set
print(f'Generating scenarios for {len(demo_days)} days (demo). Set demo_days=None for all.')

scenarios_ghi_load = generate_joint_scenarios(
    forecast_cqr, load_quantiles,
    target_days=demo_days,
    N=CFG['N_scenarios'],
)
scenarios_ghi_load.to_parquet(
    f"{CFG['output_dir']}/scenarios_joint_ghi_load_raw_{CFG['N_scenarios']}.parquet",
    index=False)
scenarios_ghi_load.head()

---
## S7: GHI → PV Conversion (PVWatts-style)

- POA_effective = GHI (minimum assumption when tilt/azimuth unknown)
- Fuentes thermal model for T_cell
- P_dc = P_dc_rated × POA/1000 × [1 + γ_pdc × (T_cell − 25)]
- P_ac = min(P_ac_rated, P_dc × inv_eff), then clip ≥ 0

In [ ]:
def ghi_to_pv(scenarios: pd.DataFrame) -> pd.DataFrame:
    """
    Convert GHI trajectories to available PV power (PVWatts-style).
    """
    sc = scenarios.copy()

    # POA = GHI (minimum assumption)
    sc['poa_effective'] = sc['ghi_wm2'].clip(lower=0)

    # Temperature model: simplified (use ambient temp estimate)
    # For each target hour, get t2m from the features if available
    # As a fallback, use a seasonal average
    sc['hour'] = sc['target_time_local'].dt.hour
    sc['month'] = sc['target_time_local'].dt.month

    # Simple ambient temp model (Taipei seasonal)
    monthly_temp = {1: 16, 2: 17, 3: 19, 4: 23, 5: 26, 6: 28,
                    7: 30, 8: 30, 9: 28, 10: 25, 11: 21, 12: 18}
    sc['temp_air'] = sc['month'].map(monthly_temp).fillna(25.0)
    sc['wind_speed_est'] = 2.0  # default

    # Simplified Fuentes/NOCT cell temperature
    # T_cell = T_ambient + POA/800 * (INOCT - 20)
    inoct = CFG['inoct']
    sc['t_cell'] = sc['temp_air'] + sc['poa_effective'] / 800.0 * (inoct - 20)

    # DC power
    p_dc_rated = CFG['pv_dc_rated_kw']
    gamma = CFG['gamma_pdc']
    sc['p_dc'] = p_dc_rated * (sc['poa_effective'] / 1000.0) * (1 + gamma * (sc['t_cell'] - 25))
    sc['p_dc'] = sc['p_dc'].clip(lower=0)

    # AC power
    p_ac_rated = CFG['pv_ac_rated_kw']
    inv_eff = CFG['inv_eff']
    sc['pv_available_kw'] = np.minimum(p_ac_rated, sc['p_dc'] * inv_eff)
    sc['pv_available_kw'] = sc['pv_available_kw'].clip(lower=0)

    # Nighttime → 0 (solar elevation check)
    site = location.Location(CFG['site_lat'], CFG['site_lon'], tz=CFG['tz'])
    times_local = pd.DatetimeIndex(sc['target_time_local']).tz_localize(CFG['tz'])
    solpos = site.get_solarposition(times_local)
    night_mask = solpos['elevation'].values <= 0
    sc.loc[night_mask, 'pv_available_kw'] = 0.0

    print(f'PV scenarios shape: {sc.shape}')
    print(f'PV range: {sc["pv_available_kw"].min():.1f} – {sc["pv_available_kw"].max():.1f} kW')
    return sc

scenarios_pv_load = ghi_to_pv(scenarios_ghi_load)
scenarios_pv_load.to_parquet(
    f"{CFG['output_dir']}/scenarios_joint_pv_load_raw_{CFG['N_scenarios']}.parquet",
    index=False)
scenarios_pv_load[['target_day_local', 'target_time_local', 'scenario_id',
                   'ghi_wm2', 'pv_available_kw', 'load_kw']].head()

---
## S8: Scenario Reduction (k-medoids)

- Reduction in PV+Load space (not GHI+Load)
- Standardise before distance calculation
- Output K medoids with probability π

In [ ]:
def reduce_scenarios(scenarios_pv: pd.DataFrame, K: int = 5) -> pd.DataFrame:
    """
    k-medoids reduction in PV+Load space per target day.
    Output: K representative scenarios with probability π.
    """
    all_reduced = []

    for day, grp in tqdm(scenarios_pv.groupby('target_day_local'), desc='k-medoids'):
        # Pivot: each scenario → 48-dim vector (24h PV + 24h Load)
        pivoted = grp.pivot_table(
            index='scenario_id',
            columns=grp['target_time_local'].dt.hour,
            values=['pv_available_kw', 'load_kw'],
            aggfunc='first'
        )
        pivoted.columns = [f'{v}_h{h}' for v, h in pivoted.columns]
        pivoted = pivoted.fillna(0)

        if len(pivoted) < K:
            continue

        # Standardise
        scaler = StandardScaler()
        X = scaler.fit_transform(pivoted.values)

        # k-medoids
        actual_K = min(K, len(pivoted))
        kmed = KMedoids(n_clusters=actual_K, random_state=CFG['random_seed'],
                        metric='euclidean', method='alternate')
        labels = kmed.fit_predict(X)

        # Medoid scenario IDs
        medoid_indices = kmed.medoid_indices_
        medoid_scenario_ids = pivoted.index[medoid_indices].tolist()

        # Probability = cluster size / total
        cluster_sizes = pd.Series(labels).value_counts().sort_index()
        probabilities = (cluster_sizes / cluster_sizes.sum()).values

        # Extract medoid rows from original scenarios
        for k_idx, (sid, prob) in enumerate(zip(medoid_scenario_ids, probabilities)):
            medoid_rows = grp[grp['scenario_id'] == sid].copy()
            medoid_rows['reduced_scenario_id'] = k_idx
            medoid_rows['probability'] = prob
            medoid_rows['medoid_raw_id'] = sid
            all_reduced.append(medoid_rows)

    reduced = pd.concat(all_reduced, ignore_index=True)

    # Select final columns per file contract
    keep_cols = [
        'issue_day_local', 'target_day_local', 'reduced_scenario_id',
        'target_time_local', 'horizon_hour',
        'pv_available_kw', 'load_kw', 'probability', 'medoid_raw_id',
    ]
    reduced = reduced[[c for c in keep_cols if c in reduced.columns]]

    # Verify Σπ = 1 per day
    pi_check = reduced.groupby('target_day_local').apply(
        lambda g: g.drop_duplicates('reduced_scenario_id')['probability'].sum())
    print(f'Reduced scenarios: {reduced.shape}')
    print(f'Probability sum per day (should be ~1.0):\n{pi_check}')

    return reduced

reduced_scenarios = reduce_scenarios(scenarios_pv_load, K=CFG['K_reduced'])
reduced_scenarios.to_parquet(
    f"{CFG['output_dir']}/scenarios_joint_pv_load_reduced_{CFG['K_reduced']}.parquet",
    index=False)
reduced_scenarios.head()

---
## Forecast Evaluation Metrics

Evaluates the CQR-calibrated GHI forecast on the **test set**:
- **Point accuracy**: MAE, RMSE on P50 median (daylight hours only)
- **Probabilistic score**: Pinball loss per quantile, CRPS proxy
- **Interval reliability**: PICP (coverage) and MPIW (sharpness) for Q10-Q90
- **Per-pair CQR coverage**: checks each symmetric pair against its nominal level

In [ ]:
def evaluate_forecast(forecast_cqr: pd.DataFrame):
    """
    Full evaluation of the CQR-calibrated forecast on the test set.
    Matches Harry's Table 4.2-1 (all hours) and Table 4.2-2 (excl nighttime zeros).
    """
    quantiles = CFG['quantiles']
    q_cols = [f'q{q:.2f}' for q in quantiles]

    test = forecast_cqr[forecast_cqr['split_name'] == 'test'].copy()
    y_all = test['label_ghi_obs_wm2'].values
    q50_all = test['q0.50'].values

    # ── 1. Point Forecast: All hours (Table 4.2-1) ──
    mae_all = np.mean(np.abs(y_all - q50_all))
    rmse_all = np.sqrt(np.mean((y_all - q50_all) ** 2))
    ss_res = np.sum((y_all - q50_all) ** 2)
    ss_tot = np.sum((y_all - np.mean(y_all)) ** 2)
    r2_all = 1 - ss_res / ss_tot if ss_tot > 0 else float('nan')

    print('=' * 55)
    print('  Table 4.2-1: Point Forecast P50 (ALL hours)')
    print('=' * 55)
    print(f'  Samples: {len(y_all)}')
    print(f'  MAE  : {mae_all:8.4f} W/m2')
    print(f'  RMSE : {rmse_all:8.4f} W/m2')
    print(f'  R2   : {r2_all:8.4f}')
    print()

    # ── 2. Point Forecast: Excl nighttime zeros (Table 4.2-2) ──
    nonzero_mask = (y_all > 0) | (q50_all > 0)
    y_nz = y_all[nonzero_mask]
    q50_nz = q50_all[nonzero_mask]
    mae_nz = np.mean(np.abs(y_nz - q50_nz))
    rmse_nz = np.sqrt(np.mean((y_nz - q50_nz) ** 2))
    ss_res_nz = np.sum((y_nz - q50_nz) ** 2)
    ss_tot_nz = np.sum((y_nz - np.mean(y_nz)) ** 2)
    r2_nz = 1 - ss_res_nz / ss_tot_nz if ss_tot_nz > 0 else float('nan')

    print('=' * 55)
    print('  Table 4.2-2: Point Forecast P50 (excl nighttime zeros)')
    print('=' * 55)
    print(f'  Samples: {len(y_nz)}')
    print(f'  MAE  : {mae_nz:8.4f} W/m2')
    print(f'  RMSE : {rmse_nz:8.4f} W/m2')
    print(f'  R2   : {r2_nz:8.4f}')
    print()

    # Use daylight-only for remaining metrics
    daylight = (test['solar_elevation'].values > 0) & nonzero_mask
    test_day = test[daylight]
    y_day = test_day['label_ghi_obs_wm2'].values

    # ── 3. Pinball Loss + CRPS proxy ──
    print('=' * 55)
    print('  PINBALL LOSS (daylight only)')
    print('=' * 55)
    pinball_losses = []
    for q, col in zip(quantiles, q_cols):
        pred = test_day[col].values
        err = y_day - pred
        loss = np.mean(np.where(err >= 0, q * err, (q - 1) * err))
        pinball_losses.append(loss)
        print(f'  {col}: {loss:8.4f}')
    crps_proxy = np.mean(pinball_losses)
    print(f'  ---')
    print(f'  CRPS proxy (avg pinball): {crps_proxy:.4f}  (lower is better)')
    print()

    # ── 4. Interval Reliability ──
    print('=' * 55)
    print('  INTERVAL RELIABILITY (daylight only)')
    print('=' * 55)

    q10 = test_day['q0.10'].values
    q90 = test_day['q0.90'].values
    picp_80 = np.mean((y_day >= q10) & (y_day <= q90)) * 100
    mpiw_80 = np.mean(q90 - q10)
    print(f'  80% interval (Q10-Q90):')
    print(f'    PICP : {picp_80:6.2f}%  (target: 80%)')
    print(f'    MPIW : {mpiw_80:8.2f} W/m2')
    print()

    q05 = test_day['q0.05'].values
    q95 = test_day['q0.95'].values
    picp_90 = np.mean((y_day >= q05) & (y_day <= q95)) * 100
    mpiw_90 = np.mean(q95 - q05)
    print(f'  90% interval (Q05-Q95):')
    print(f'    PICP : {picp_90:6.2f}%  (target: 90%)')
    print(f'    MPIW : {mpiw_90:8.2f} W/m2')
    print()

    # ── 5. Per-pair CQR coverage ──
    print('=' * 55)
    print('  PER-PAIR CQR COVERAGE (daylight only)')
    print('=' * 55)
    print(f'  {"Pair":>14s}  {"Nominal":>8s}  {"Actual":>8s}  {"Status"}')
    print(f'  ' + '-' * 48)
    for tau_low, tau_high in CFG['quantile_pairs']:
        col_low = f'q{tau_low:.2f}'
        col_high = f'q{tau_high:.2f}'
        nominal = (tau_high - tau_low) * 100
        actual = np.mean((y_day >= test_day[col_low].values) &
                         (y_day <= test_day[col_high].values)) * 100
        diff = actual - nominal
        status = 'OK' if abs(diff) < 5 else ('UNDER' if diff < 0 else 'OVER')
        print(f'  ({tau_low:.2f}, {tau_high:.2f})  {nominal:7.1f}%  {actual:7.1f}%  {status}')
    print()

    # ── 6. Per-quantile empirical coverage ──
    print('=' * 55)
    print('  EMPIRICAL COVERAGE (daylight only)')
    print('=' * 55)
    for q, col in zip(quantiles, q_cols):
        cov = np.mean(y_day <= test_day[col].values) * 100
        diff = cov - q * 100
        flag = '' if abs(diff) < 5 else f'  <-- off by {diff:+.1f}pp'
        print(f'  {col}: {cov:6.1f}%  (nominal {q*100:.0f}%){flag}')

    return {
        'mae_all': mae_all, 'rmse_all': rmse_all, 'r2_all': r2_all,
        'mae_nz': mae_nz, 'rmse_nz': rmse_nz, 'r2_nz': r2_nz,
        'crps_proxy': crps_proxy,
        'picp_80': picp_80, 'mpiw_80': mpiw_80,
        'picp_90': picp_90, 'mpiw_90': mpiw_90,
    }

eval_results = evaluate_forecast(forecast_cqr)

---
## S9: QA & Metadata

In [ ]:
def generate_qa_report(forecast_cqr, reduced_scenarios, dataset) -> dict:
    """Generate QA report per spec §6.4."""
    quantiles = CFG['quantiles']
    q_cols = [f'q{q:.2f}' for q in quantiles]

    test_fc = forecast_cqr[forecast_cqr['split_name'] == 'test']

    qa = {
        'gate_compliance': {
            'all_chosen_init_before_deadline': True,  # enforced by construction
        },
        'data_qa': {
            'cwa_ghi_unit': 'W/m2 (converted from MJ/m2)',
            'nwp_tp_method': 'TP-A (3h evenly distributed)',
            'total_features_hourly_rows': len(dataset),
        },
        'forecast_qa': {
            'test_issue_days': int(test_fc['target_day_local'].nunique()),
            'crossing_count': int((np.diff(test_fc[q_cols].values, axis=1) < 0).sum()),
        },
        'reduction_qa': {
            'N_raw': CFG['N_scenarios'],
            'K_reduced': CFG['K_reduced'],
        },
    }

    # Coverage check (per quantile)
    if 'label_ghi_obs_wm2' in test_fc.columns:
        y_true = test_fc['label_ghi_obs_wm2'].values
        coverage = {}
        for q, col in zip(quantiles, q_cols):
            cov = (y_true <= test_fc[col].values).mean()
            coverage[col] = round(float(cov), 4)
        qa['forecast_qa']['empirical_coverage'] = coverage

        # Pinball loss
        pinball = {}
        for q, col in zip(quantiles, q_cols):
            pred = test_fc[col].values
            err = y_true - pred
            loss = np.where(err >= 0, q * err, (q - 1) * err).mean()
            pinball[col] = round(float(loss), 4)
        qa['forecast_qa']['pinball_loss'] = pinball

    # Probability sum check
    if len(reduced_scenarios) > 0:
        pi_sums = reduced_scenarios.groupby('target_day_local').apply(
            lambda g: g.drop_duplicates('reduced_scenario_id')['probability'].sum())
        qa['reduction_qa']['probability_sum_range'] = (
            f"{pi_sums.min():.4f} – {pi_sums.max():.4f}")

    return qa

qa_report = generate_qa_report(forecast_cqr, reduced_scenarios, dataset)

with open(f"{CFG['output_dir']}/qa_report.json", 'w') as f:
    json.dump(qa_report, f, indent=2, default=str)

print(json.dumps(qa_report, indent=2, default=str))

In [ ]:
def generate_run_metadata() -> dict:
    """Generate reproducibility metadata."""
    meta = {
        'run_id': hashlib.md5(str(datetime.now()).encode()).hexdigest()[:12],
        'created_at': datetime.now().isoformat(),
        'timezone': CFG['tz'],
        'random_seed': CFG['random_seed'],
        'N_scenarios': CFG['N_scenarios'],
        'K_reduced': CFG['K_reduced'],
        'quantiles': CFG['quantiles'],
        'site_metadata': {
            'lat': CFG['site_lat'],
            'lon': CFG['site_lon'],
            'elevation': CFG['site_elev'],
            'name': 'NTUST',
        },
        'pv_params': {
            'pv_dc_rated_kw': CFG['pv_dc_rated_kw'],
            'gamma_pdc': CFG['gamma_pdc'],
            'inv_eff': CFG['inv_eff'],
            'dc_ac_ratio': CFG['dc_ac_ratio'],
            'inoct': CFG['inoct'],
        },
        'config_hash': hashlib.md5(json.dumps(CFG, default=str).encode()).hexdigest(),
    }
    return meta

run_meta = generate_run_metadata()
with open(f"{CFG['output_dir']}/run_metadata.json", 'w') as f:
    json.dump(run_meta, f, indent=2, default=str)

print(json.dumps(run_meta, indent=2, default=str))

---
## Summary of Output Artifacts

| File | Purpose |
|------|--------|
| `features_hourly.parquet` | Merged CWA+NWP hourly features |
| `nwp_gate_manifest.parquet` | NWP gate traceability |
| `dataset_issue_target.parquet` | Supervised training dataset |
| `split_manifest.parquet` | Chronological split record |
| `forecast_ghi_quantiles_daily_base_raw.parquet` | Raw XGBQ 19Q |
| `forecast_ghi_quantiles_daily.parquet` | CQR-calibrated 19Q (official) |
| `load_quantiles_hourly.parquet` | Load marginals |
| `scenarios_joint_ghi_load_raw_N.parquet` | Raw joint scenarios |
| `scenarios_joint_pv_load_raw_N.parquet` | After GHI→PV |
| `scenarios_joint_pv_load_reduced_K.parquet` | **MILP input** |
| `qa_report.json` | QA summary |
| `run_metadata.json` | Reproducibility metadata |

In [ ]:
# List all output files
output_dir = Path(CFG['output_dir'])
for f in sorted(output_dir.glob('*')):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:55s} {size_mb:8.2f} MB')

In [ ]:
print('Pipeline complete.')
print(f'MILP input file: {CFG["output_dir"]}/scenarios_joint_pv_load_reduced_{CFG["K_reduced"]}.parquet')
print(f'Each issue day has 24h x {CFG["K_reduced"]} scenarios with probability pi summing to 1.')